# Underwater Ship Acoustic EDA
**Task 1** Ship Type Classification (4-class Macro F1) / **Task 2** Ship ID Retrieval (Recall@K)

Data path: `../data/` (relative to project root)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.gridspec as gridspec
import librosa
import librosa.display
import warnings
warnings.filterwarnings('ignore')

# ── Korean font fix (Windows) ─────────────────────────────────────────────────
for _fp in ['C:/Windows/Fonts/malgun.ttf', 'C:/Windows/Fonts/malgunbd.ttf']:
    if os.path.exists(_fp):
        fm.fontManager.addfont(_fp)
        plt.rcParams['font.family'] = fm.FontProperties(fname=_fp).get_name()
        print(f'font: {plt.rcParams["font.family"]}')
        break
plt.rcParams['axes.unicode_minus'] = False
# ─────────────────────────────────────────────────────────────────────────────

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

BASE = '../data'
CLASSES = ['A_SmallWorking', 'B_MotorBoat', 'C_Passenger', 'D_LargeShip']
COLORS  = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
print('ready')

## 1. Data Loading

In [ ]:
train     = pd.read_csv(f'{BASE}/train/train.csv')
t1_val    = pd.read_csv(f'{BASE}/task1_test/val.csv')
t1_test   = pd.read_csv(f'{BASE}/task1_test/test.csv')
gallery   = pd.read_csv(f'{BASE}/task2_test/gallery.csv')
t2_val    = pd.read_csv(f'{BASE}/task2_test/val.csv')
t2_test   = pd.read_csv(f'{BASE}/task2_test/test.csv')
ship_list = pd.read_csv(f'{BASE}/ship_list.csv')
tgt_ships = pd.read_csv(f'{BASE}/task2_target_ships.csv')

TRAIN_AUDIO   = f'{BASE}/train/audio'
TASK1_AUDIO   = f'{BASE}/task1_test/audio'
TASK2_AUDIO   = f'{BASE}/task2_test/audio'

print('=== Dataset Sizes ===')
for name, df in [('train', train), ('t1_val', t1_val), ('t1_test', t1_test),
                 ('gallery', gallery), ('t2_val', t2_val), ('t2_test', t2_test)]:
    print(f'  {name:10s}: {len(df):6,} rows  cols={df.columns.tolist()}')

## 2. Basic Checks — Missing Values, Types, Statistics

In [ ]:
print('=== train dtypes ===')
print(train.dtypes)
print()

print('=== Missing Values (train) ===')
miss = train.isnull().sum()
print(miss[miss > 0] if miss.any() else 'None')
print()

print('=== AIS Basic Statistics (train) ===')
train[['sog', 'cog', 'true_heading']].describe().round(2)

## 3. Task 1 — Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (df, title) in zip(axes, [(train, 'Train (35,183)'), (t1_val, 'Val (11,079)')]):
    counts = df['ship_type'].value_counts().reindex(CLASSES).fillna(0).astype(int)
    bars = ax.bar(CLASSES, counts.values, color=COLORS, edgecolor='white', linewidth=0.8)
    for bar, cnt in zip(bars, counts.values):
        pct = cnt / len(df) * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
                f'{cnt:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Clip Count')
    ax.set_ylim(0, counts.max() * 1.22)
    ax.set_xticklabels(CLASSES, rotation=20, ha='right')
    ax.grid(axis='y', alpha=0.3)

imb = train['ship_type'].value_counts()
print(f'Imbalance ratio: {imb.max()/imb.min():.2f}x  -> class_weight or Focal Loss recommended')
plt.suptitle('Task 1 — Ship Type Distribution', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Clips per Ship Distribution (Leakage Risk Check)

In [ ]:
clips_per_ship = train.groupby('ship_id').size().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(clips_per_ship.values, bins=40, color='#4C72B0', edgecolor='white')
axes[0].axvline(clips_per_ship.mean(), color='red', linestyle='--', label=f'mean={clips_per_ship.mean():.0f}')
axes[0].set_xlabel('Clips per Ship')
axes[0].set_ylabel('Ship Count')
axes[0].set_title('Clips per Ship Distribution (train, 362 ships)')
axes[0].legend()
axes[0].grid(alpha=0.3)

clip_by_ship = train.groupby(['ship_id', 'ship_type']).size().reset_index(name='clips')
groups = [clip_by_ship[clip_by_ship['ship_type'] == c]['clips'].values for c in CLASSES]
bp = axes[1].boxplot(groups, labels=CLASSES, patch_artist=True)
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Clips per Ship by Class')
axes[1].set_ylabel('Clips per Ship')
axes[1].set_xticklabels(CLASSES, rotation=20, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
print(f'min={clips_per_ship.min()}, max={clips_per_ship.max()}, mean={clips_per_ship.mean():.1f}')
print('-> GroupKFold required: same ship clips in train/val = score inflation')

## 5. AIS Feature Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

ax = axes[0, 0]
sog = train['sog'].clip(0, 30)
ax.hist(sog, bins=50, color='#4C72B0', edgecolor='white')
zero_pct = (train['sog'] == 0).mean() * 100
ax.set_title(f'SOG Distribution (SOG=0: {zero_pct:.1f}%)')
ax.set_xlabel('Speed (knots, clipped at 30)')
ax.set_ylabel('Clip Count')
ax.grid(alpha=0.3)

ax = axes[0, 1]
groups_sog = [train[train['ship_type'] == c]['sog'].clip(0, 30).values for c in CLASSES]
bp = ax.boxplot(groups_sog, labels=CLASSES, patch_artist=True)
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_title('SOG by Class')
ax.set_ylabel('SOG (knots)')
ax.set_xticklabels(CLASSES, rotation=20, ha='right')
ax.grid(axis='y', alpha=0.3)

ax = axes[0, 2]
for c, color in zip(CLASSES, COLORS):
    sub = train[train['ship_type'] == c]
    ax.hist(sub['cog'], bins=36, alpha=0.5, label=c, color=color, density=True)
ax.set_title('COG Distribution by Class')
ax.set_xlabel('COG (degrees)')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

ax = plt.subplot(2, 3, 4, projection='polar')
ax.hist(np.deg2rad(train['cog'].values), bins=36, color='#4C72B0', alpha=0.7)
ax.set_title('COG Direction', pad=15)

ax = plt.subplot(2, 3, 5, projection='polar')
ax.hist(np.deg2rad(train['true_heading'].values), bins=36, color='#DD8452', alpha=0.7)
ax.set_title('True Heading Distribution', pad=15)

ax = axes[1, 2]
sog_nz = train[train['sog'] > 0]
groups_sog2 = [sog_nz[sog_nz['ship_type'] == c]['sog'].clip(0, 30).values for c in CLASSES]
vp = ax.violinplot(groups_sog2, positions=range(4), showmedians=True)
for pc, color in zip(vp['bodies'], COLORS):
    pc.set_facecolor(color); pc.set_alpha(0.6)
ax.set_xticks(range(4))
ax.set_xticklabels(CLASSES, rotation=20, ha='right')
ax.set_title('SOG > 0 Only (Underway)')
ax.set_ylabel('SOG (knots)')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('AIS Feature Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Time Distribution (Recording Period)

In [ ]:
train['ts'] = pd.to_datetime(train['ais_timestamp'], format='mixed', utc=True)
t1_val['ts'] = pd.to_datetime(t1_val['ais_timestamp'], format='mixed', utc=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
train['month'] = train['ts'].dt.to_period('M').astype(str)
mc = train.groupby('month').size()
ax.bar(mc.index, mc.values, color='#4C72B0', edgecolor='white')
ax.set_title('Monthly Clip Count (Train)')
ax.set_xlabel('Year-Month')
ax.set_ylabel('Clip Count')
ax.set_xticklabels(mc.index, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
for c, color in zip(CLASSES, COLORS):
    sub = train[train['ship_type'] == c]
    mc2 = sub.groupby('month').size()
    ax.plot(mc2.index, mc2.values, label=c, color=color, marker='o', markersize=3)
ax.set_title('Monthly Clips by Class')
ax.set_xlabel('Year-Month')
ax.set_ylabel('Clip Count')
ax.set_xticklabels(mc.index, rotation=45, ha='right')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

print(f'Recording period: {train["ts"].min().date()} ~ {train["ts"].max().date()}')
plt.tight_layout()
plt.show()

## 7. Audio Waveforms — Samples by Class

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

for ax, cls, color in zip(axes, CLASSES, COLORS):
    fn = train[train['ship_type'] == cls].iloc[0]['filename']
    path = os.path.join(TRAIN_AUDIO, fn)
    y, sr = librosa.load(path, sr=32000, duration=5.0)
    t = np.linspace(0, len(y)/sr, len(y))
    ax.plot(t, y, color=color, linewidth=0.6, alpha=0.8)
    ax.set_ylabel(cls, fontsize=9)
    ax.set_ylim(-0.2, 0.2)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Waveforms by Class (First Sample)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Log-mel Spectrograms — Samples by Class

In [ ]:
SR, N_MELS, N_FFT, HOP = 32000, 128, 2048, 512
F_MIN, F_MAX = 50, 16000

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for col, (cls, color) in enumerate(zip(CLASSES, COLORS)):
    rows = train[train['ship_type'] == cls].head(2)
    for row_idx, (_, rec) in enumerate(rows.iterrows()):
        ax = axes[row_idx][col]
        path = os.path.join(TRAIN_AUDIO, rec['filename'])
        y, _ = librosa.load(path, sr=SR, duration=5.0)
        mel = librosa.feature.melspectrogram(
            y=y, sr=SR, n_mels=N_MELS, n_fft=N_FFT,
            hop_length=HOP, fmin=F_MIN, fmax=F_MAX
        )
        log_mel = librosa.power_to_db(mel, ref=np.max)
        img = librosa.display.specshow(
            log_mel, sr=SR, hop_length=HOP,
            x_axis='time', y_axis='mel',
            fmin=F_MIN, fmax=F_MAX, ax=ax, cmap='magma'
        )
        sog_val = rec.get('sog', '?')
        ax.set_title(f'{cls}\nSOG={sog_val}kt', fontsize=8, color=color, fontweight='bold')
        if col > 0:
            ax.set_ylabel('')
        if row_idx == 0:
            ax.set_xlabel('')

plt.colorbar(img, ax=axes, format='%+2.0f dB')
plt.suptitle('Log-mel Spectrograms — 2 Samples per Class (128mel, 32kHz)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Frequency Energy Distribution — Mean Spectrum by Class

In [ ]:
N_SAMPLE = 30

fig, ax = plt.subplots(figsize=(12, 5))

for cls, color in zip(CLASSES, COLORS):
    rows = train[train['ship_type'] == cls].sample(N_SAMPLE, random_state=42)
    specs = []
    for _, rec in rows.iterrows():
        path = os.path.join(TRAIN_AUDIO, rec['filename'])
        y, _ = librosa.load(path, sr=SR, duration=5.0)
        mel = librosa.feature.melspectrogram(
            y=y, sr=SR, n_mels=N_MELS, n_fft=N_FFT,
            hop_length=HOP, fmin=F_MIN, fmax=F_MAX
        )
        specs.append(librosa.power_to_db(mel, ref=np.max).mean(axis=1))
    mean_spec = np.mean(specs, axis=0)
    mel_freqs = librosa.mel_frequencies(n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX)
    ax.plot(mel_freqs, mean_spec, label=cls, color=color, linewidth=2, alpha=0.85)

ax.set_xlabel('Frequency (Hz, mel scale)')
ax.set_ylabel('Mean Energy (dB)')
ax.set_title(f'Mean Frequency Spectrum by Class (avg {N_SAMPLE} samples each)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Task 2 — Gallery Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
gal_cpp = gallery.groupby('ship_id').size().sort_values()
ax.hist(gal_cpp.values, bins=25, color='#55A868', edgecolor='white')
ax.axvline(gal_cpp.mean(), color='red', linestyle='--', label=f'mean={gal_cpp.mean():.0f}')
ax.set_title(f'Gallery Clips per Ship (100 ships)\nmin={gal_cpp.min()}, max={gal_cpp.max()}')
ax.set_xlabel('Clips per Ship')
ax.set_ylabel('Ship Count')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
gc = gallery['ship_type'].value_counts().reindex(CLASSES).fillna(0).astype(int)
ax.bar(CLASSES, gc.values, color=COLORS, edgecolor='white')
for i, v in enumerate(gc.values):
    ax.text(i, v + 5, str(v), ha='center', fontsize=9)
ax.set_title('Gallery Ship Type Distribution')
ax.set_ylabel('Clip Count')
ax.set_xticklabels(CLASSES, rotation=20, ha='right')
ax.grid(axis='y', alpha=0.3)

ax = axes[2]
t2v_cpp = t2_val.groupby('ship_id').size().sort_values()
ax.hist(t2v_cpp.values, bins=20, color='#C44E52', edgecolor='white')
ax.set_title(f'Task2 Val Queries per Ship\n(100 ships, {len(t2_val):,} clips total)')
ax.set_xlabel('Query Clips per Ship')
ax.set_ylabel('Ship Count')
ax.grid(alpha=0.3)

plt.suptitle('Task 2 Data Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('=== Task2 test ===')
print(f'  columns: {t2_test.columns.tolist()}  ->  No AIS! Audio-only retrieval required')
print(f'  query count: {len(t2_test):,}')

## 11. Data Leakage Check

In [ ]:
print('=== Leakage Check ===\n')

print(f'[1] Does t1_val have ship_id column? {"ship_id" in t1_val.columns}')
print('    -> If not, safe (cannot reverse-lookup ship ID)')

train_ids  = set(train['ship_id'].unique())
gallery_ids = set(gallery['ship_id'].unique())
overlap = train_ids & gallery_ids
print(f'\n[2] train ∩ gallery ship_id overlap: {len(overlap)} ships / {len(gallery_ids)} gallery ships')
print('    -> Overlapping ships can be learned from train (expected, closed-set structure)')

print(f'\n[3] train 362 ships clips per ship: min={train.groupby("ship_id").size().min()}, max={train.groupby("ship_id").size().max()}')
print('    -> Random KFold: same ship clips appear in both train/val -> OOF score inflation')
print('    -> Must use GroupKFold(group=ship_id)')

train_fns = set(train['filename'])
val_fns   = set(t1_val['filename'])
fn_overlap = train_fns & val_fns
print(f'\n[4] train ∩ t1_val filename overlap: {len(fn_overlap)}')
print('    -> 0 means no file-level leakage')

## 12. Key Summary

In [ ]:
print('=' * 60)
print('EDA Key Summary')
print('=' * 60)

imb = train['ship_type'].value_counts()
print(f"""
[Task 1 — Ship Type Classification]
  · Train: {len(train):,} clips / Val: {len(t1_val):,} clips / Test: {len(t1_test):,} clips
  · Class imbalance: {imb.max()/imb.min():.1f}x  -> class_weight or Focal Loss required
  · Train/val distribution similar -> no covariate shift
  · SOG=0 ratio {(train['sog']==0).mean()*100:.0f}%  -> AIS features alone are unreliable
  · GroupKFold(ship_id) required (362 ships, avg ~97 clips/ship)

[Task 2 — Ship ID Retrieval]
  · Gallery: 100 ships / {len(gallery):,} clips  (high variance: min={gallery.groupby('ship_id').size().min()}, max={gallery.groupby('ship_id').size().max()})
  · Val queries: {len(t2_val):,} clips  (ship_id, ship_type available -> self-validation possible)
  · Test queries: {len(t2_test):,} clips  (filename only! No AIS)
  · Retrieval model must work on audio embeddings alone

[Audio]
  · 5s / 32kHz / mono / 24bit  -> fixed length, simple preprocessing
  · Log-mel (128mel, n_fft=2048, hop=512) recommended
  · Low-frequency energy difference is key for ship type separation

[Next Steps]
  1. Task 1 baseline: eca_nfnet_l0 + class_weight
  2. Task 2 baseline: same backbone, 362-class classification -> extract embeddings -> cosine retrieval
""")
print('=' * 60)